In [1]:
from preprocessing import Preprocessor
from analysis import Analysis, EIS_Analysis
import pandas as pd


In [2]:
filepath = '/Users/maxtenenbaum/Desktop/20240717_IF0003_IIA31_B_PEDOT_E01_EIS.DTA'

In [3]:
# Loads filepath into preprocessor
preprocessing = Preprocessor(filepath)

# Loading EIS analysis class
eis_analysis = EIS_Analysis()


In [4]:
# Checks if EIS
preprocessing.pull_eis_metadata()

In [5]:
def new_extract_eis():
        try:
            with open(filepath, 'r', encoding='utf-8') as file:
                lines = file.readlines()
        except UnicodeDecodeError:
            with open(filepath, 'r', encoding='ISO-8859-1') as file:
                lines = file.readlines()

        eis_data = []
        headers = []
        data_section = False

        for i, line in enumerate(lines):
            if 'ZCURVE' in line:
                data_section = True
                continue
            if data_section and line.startswith('\tPt'):
                headers = line.strip().split('\t')[1:]  
                data_section = False  
            elif line.startswith('\t'):
                values = line.strip().split('\t')[1:]
                if len(values) == len(headers):
                    eis_data.append(values)

        eis_df = pd.DataFrame(eis_data, columns=headers)
        eis_df = eis_df.apply(pd.to_numeric, errors='ignore')
        #eis_dataframe = eis_df[['Freq', 'Zreal', 'Zmod', 'Zphz']]

        return eis_df

In [37]:
dataframe = new_extract_eis()

In [42]:
frequencies = [1.0, 1000.0, 100000.0, 1000000.0]
closest_frequencies = eis_analysis.get_closest_freq(dataframe, frequencies)
filtered_df = dataframe[dataframe["Freq"].isin(closest_frequencies)]

In [68]:
new_df = pd.DataFrame(' ', index=dataframe.index, columns=dataframe.columns)
new_df.iloc[:len(filtered_df)] = filtered_df.values

In [98]:

new_df['Zmod'] = dataframe['Zmod'].iloc[2:].reset_index(drop=True)
new_df['Zphz'] = dataframe['Zphz'].iloc[2:].reset_index(drop=True)


In [99]:
print(new_df)

   Time      Freq     Zreal      Zimag Zsig      Zmod       Zphz  \
0     3  100019.5  5024.838  -255.4786    1  5031.328  -2.910592   
1    25  1000.702   6551.32  -2748.913    1   5052.84  -2.920468   
2    68  0.997765  177734.8  -323842.6    1  5082.216  -3.114081   
3                                            5117.985  -3.504335   
4                                            5188.028  -2.960445   
5                                            5230.285  -3.899679   
6                                            5293.212  -5.015213   
7                                            5371.562   -6.25629   
8                                             5476.43  -7.705872   
9                                             5613.83  -9.438905   
10                                           5781.479  -11.40684   
11                                           5994.795  -13.67329   
12                                            6272.01  -16.33351   
13                                           662